# Expériences sur Secret Env 2

**Environnement boîte noire fourni par le prof.**

- `num_states` = 2 097 152 (2^21)
- `num_actions` = 3
- `num_rewards` = 3 → valeurs {-1.0, 0.0, +1.0}

## Pourquoi pas de Dynamic Programming ?

La matrice de transition dense `p[s, a, s', r]` ferait :

$$2{,}097{,}152 \times 3 \times 2{,}097{,}152 \times 3 \approx 4 \times 10^{13} \text{ floats} \approx 316 \text{ TB}$$

→ **infaisable en mémoire**. On utilise donc uniquement les méthodes **model-free** (MC, TD, Dyna-Q) qui apprennent en interagissant avec l'environnement, sans avoir à construire tout le MDP.

Note : la table Q de taille `|S| × |A|` = 6 291 456 flottants ≈ 48 Mo — largement gérable.

In [21]:
import os, sys, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.append('..')

from environments.secret_env_2 import SecretEnv2
from algorithms.monte_carlo import (
    monte_carlo_es,
    on_policy_first_visit_mc_control,
    off_policy_mc_control,
)
from algorithms.temporal_difference_learning import sarsa, q_learning
from algorithms.planning import dyna_q, dyna_q_plus

SAVE_DIR = '../models/secret_env_2'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Imports OK')

Imports OK


## 1. Exploration de l'environnement

Un épisode aléatoire pour voir la structure : combien de steps par épisode, quels rewards sont récoltés.

In [22]:
env = SecretEnv2()
print(f'nb_states  = {env.max_state_count():,}')
print(f'nb_actions = {env.max_actions_count()}')
print(f'nb_rewards = {env._nb_rewards}')
print(f'rewards    = {[env._raw.reward(i) for i in range(env._nb_rewards)]}')

np.random.seed(0)
lengths, finals = [], []
for _ in range(200):
    env.reset()
    steps = 0
    while not env.is_game_over() and steps < 500:
        env.step(np.random.choice(env.available_actions()))
        steps += 1
    lengths.append(steps)
    finals.append(env.score())

print(f'\nÉpisodes aléatoires (200 runs):')
print(f'  longueur moyenne = {np.mean(lengths):.1f} (min={min(lengths)}, max={max(lengths)})')
print(f'  score moyen      = {np.mean(finals):+.3f}')
print(f'  score min / max  = {min(finals):+.2f} / {max(finals):+.2f}')

nb_states  = 2,097,152
nb_actions = 3
nb_rewards = 3
rewards    = [-1.0, 0.0, 1.0]

Épisodes aléatoires (200 runs):
  longueur moyenne = 255.0 (min=255, max=255)
  score moyen      = -59.915
  score min / max  = -67.00 / -50.00


## 2. Baseline : politique aléatoire

Pour comparer, on garde en tête le score moyen d'un agent qui joue au hasard.

In [23]:
def evaluate_policy(env, pi, n_episodes=200, max_steps=500, seed=42):
    """Rejoue la policy et retourne le score moyen."""
    np.random.seed(seed)
    scores = []
    for _ in range(n_episodes):
        env.reset()
        steps = 0
        while not env.is_game_over() and steps < max_steps:
            s = env.current_state()
            available = env.available_actions()
            # argmax restreint aux actions disponibles
            a = max(available, key=lambda x: pi[s, x])
            env.step(a)
            steps += 1
        scores.append(env.score())
    return np.mean(scores), np.std(scores)

def evaluate_random(env, n_episodes=200, max_steps=500, seed=42):
    np.random.seed(seed)
    scores = []
    for _ in range(n_episodes):
        env.reset()
        steps = 0
        while not env.is_game_over() and steps < max_steps:
            env.step(np.random.choice(env.available_actions()))
            steps += 1
        scores.append(env.score())
    return np.mean(scores), np.std(scores)

mean_r, std_r = evaluate_random(env)
print(f'Baseline aléatoire : score moyen = {mean_r:+.3f} ± {std_r:.3f}')

Baseline aléatoire : score moyen = -59.160 ± 4.686


## 3. On-policy First-Visit Monte Carlo

Attention : avec 2M états, la plupart ne seront jamais visités. Une seule table Q est utilisée pour tout retenir.

In [24]:
t = time.time()
pi_mc, Q_mc = on_policy_first_visit_mc_control(
    env, gamma=0.999, epsilon=0.1, num_episodes=5000, max_steps=200
)
t_mc = time.time() - t
mean_mc, std_mc = evaluate_policy(env, pi_mc)
print(f'MC (5k épisodes) : temps={t_mc:.1f}s | score={mean_mc:+.3f} ± {std_mc:.3f}')

np.save(f'{SAVE_DIR}/mc_pi.npy', pi_mc)
np.save(f'{SAVE_DIR}/mc_Q.npy', Q_mc)
print(f'  → sauvegardé dans {SAVE_DIR}/mc_*.npy')

MC (5k épisodes) : temps=10.4s | score=-68.000 ± 0.000
  → sauvegardé dans ../models/secret_env_2/mc_*.npy


## 4. Sarsa

In [25]:
t = time.time()
pi_sarsa, Q_sarsa = sarsa(
    env, gamma=0.999, alpha=0.1, epsilon=0.1,
    num_episodes=5000, max_steps=200
)
t_sarsa = time.time() - t
mean_s, std_s = evaluate_policy(env, pi_sarsa)
print(f'Sarsa (5k épisodes) : temps={t_sarsa:.1f}s | score={mean_s:+.3f} ± {std_s:.3f}')

np.save(f'{SAVE_DIR}/sarsa_pi.npy', pi_sarsa)
np.save(f'{SAVE_DIR}/sarsa_Q.npy', Q_sarsa)

Sarsa (5k épisodes) : temps=11.0s | score=-50.000 ± 0.000


## 5. Q-Learning

In [26]:
t = time.time()
pi_ql, Q_ql = q_learning(
    env, gamma=0.999, alpha=0.1, epsilon=0.1,
    num_episodes=5000, max_steps=200
)
t_ql = time.time() - t
mean_q, std_q = evaluate_policy(env, pi_ql)
print(f'Q-Learning (5k épisodes) : temps={t_ql:.1f}s | score={mean_q:+.3f} ± {std_q:.3f}')

np.save(f'{SAVE_DIR}/qlearning_pi.npy', pi_ql)
np.save(f'{SAVE_DIR}/qlearning_Q.npy', Q_ql)

Q-Learning (5k épisodes) : temps=14.9s | score=-50.000 ± 0.000


## 6. Dyna-Q

In [27]:
t = time.time()
Q_dyna, pi_dyna = dyna_q(
    env, gamma=0.999, alpha=0.1, epsilon=0.1,
    max_steps=50_000, N=5
)
t_dyna = time.time() - t
mean_d, std_d = evaluate_policy(env, pi_dyna)
print(f'Dyna-Q (50k steps réels, N=5) : temps={t_dyna:.1f}s | score={mean_d:+.3f} ± {std_d:.3f}')

np.save(f'{SAVE_DIR}/dynaq_pi.npy', pi_dyna)
np.save(f'{SAVE_DIR}/dynaq_Q.npy', Q_dyna)

Dyna-Q (50k steps réels, N=5) : temps=5.8s | score=-56.000 ± 0.000


## 6bis. Monte Carlo Exploring Starts (variante safe)

Variante `monte_carlo_es_safe` qui restreint l'action greedy à `env.available_actions()`.

In [28]:
results = {
    'Random':     (mean_r, std_r, 0.0),
    'MC (on-pol)':(mean_mc, std_mc, t_mc),
    'MC ES':      (mean_es, std_es, t_es),
    'Sarsa':      (mean_s, std_s, t_sarsa),
    'Q-Learning': (mean_q, std_q, t_ql),
    'Dyna-Q':     (mean_d, std_d, t_dyna),
}

print(f"{'Algo':<14} {'Score':>10} {'Std':>8} {'Temps (s)':>12}")
print('-' * 47)
for name, (m, s, t) in results.items():
    print(f'{name:<14} {m:>+10.3f} {s:>8.3f} {t:>12.1f}')

names = list(results.keys())
means = [results[n][0] for n in names]
stds  = [results[n][1] for n in names]

plt.figure(figsize=(10, 5))
plt.bar(names, means, yerr=stds, capsize=6,
        color=['gray','#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'])
plt.axhline(0, color='black', linewidth=0.5)
plt.title('Secret Env 2 — score moyen par algorithme (200 évaluations)')
plt.ylabel('Score moyen')
plt.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('secret_env_2_comparison.png', dpi=100)
plt.close()
print('\nFigure -> secret_env_2_comparison.png')

Algo                Score      Std    Temps (s)
-----------------------------------------------
Random            -59.160    4.686          0.0
MC (on-pol)       -68.000    0.000         10.4
MC ES             -65.000    0.000          7.8
Sarsa             -50.000    0.000         11.0
Q-Learning        -50.000    0.000         14.9
Dyna-Q            -56.000    0.000          5.8

Figure -> secret_env_2_comparison.png


## 7. Comparaison globale

In [29]:
results = {
    'Random':     (mean_r, std_r, 0.0),
    'MC':         (mean_mc, std_mc, t_mc),
    'Sarsa':      (mean_s, std_s, t_sarsa),
    'Q-Learning': (mean_q, std_q, t_ql),
    'Dyna-Q':     (mean_d, std_d, t_dyna),
}

print(f"{'Algo':<12} {'Score':>10} {'Std':>8} {'Temps (s)':>12}")
print('-' * 45)
for name, (m, s, t) in results.items():
    print(f'{name:<12} {m:>+10.3f} {s:>8.3f} {t:>12.1f}')

names = list(results.keys())
means = [results[n][0] for n in names]
stds  = [results[n][1] for n in names]

plt.figure(figsize=(9, 5))
plt.bar(names, means, yerr=stds, capsize=6, color=['gray','#4C72B0','#55A868','#C44E52','#8172B2'])
plt.axhline(0, color='black', linewidth=0.5)
plt.title('Secret Env 2 — score moyen par algorithme (200 évaluations)')
plt.ylabel('Score moyen')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('secret_env_2_comparison.png', dpi=100)
plt.close()
print('\nFigure -> secret_env_2_comparison.png')

Algo              Score      Std    Temps (s)
---------------------------------------------
Random          -59.160    4.686          0.0
MC              -68.000    0.000         10.4
Sarsa           -50.000    0.000         11.0
Q-Learning      -50.000    0.000         14.9
Dyna-Q          -56.000    0.000          5.8

Figure -> secret_env_2_comparison.png


## 8. Étude d'hyperparamètres (epsilon pour Q-Learning)

In [30]:
epsilons = [0.01, 0.05, 0.1, 0.2, 0.3]
scores_by_eps = []
for eps in epsilons:
    pi, Q = q_learning(env, gamma=0.999, alpha=0.1, epsilon=eps,
                       num_episodes=2000, max_steps=200)
    m, s = evaluate_policy(env, pi, n_episodes=100)
    scores_by_eps.append((m, s))
    print(f'epsilon={eps:.2f} → score={m:+.3f} ± {s:.3f}')

ms = [x[0] for x in scores_by_eps]
ss = [x[1] for x in scores_by_eps]
plt.figure(figsize=(8, 4))
plt.errorbar(epsilons, ms, yerr=ss, marker='o', capsize=5)
plt.title('Secret Env 2 — impact d\'epsilon sur Q-Learning')
plt.xlabel('epsilon')
plt.ylabel('Score moyen (100 évaluations)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('secret_env_2_epsilon.png', dpi=100)
plt.close()
print('Figure -> secret_env_2_epsilon.png')

epsilon=0.01 → score=-42.000 ± 0.000
epsilon=0.05 → score=-42.000 ± 0.000
epsilon=0.10 → score=-44.000 ± 0.000
epsilon=0.20 → score=-46.000 ± 0.000
epsilon=0.30 → score=-44.000 ± 0.000
Figure -> secret_env_2_epsilon.png


## 9. Conclusion pour Secret Env 2

*À compléter après avoir vu les résultats* :
- Meilleur algorithme : 
- Meilleurs hyperparamètres : 
- Observations sur l'environnement (durée moyenne des épisodes, sparsité des rewards, etc.) :

Les policies et Q apprises sont sauvegardées dans `../models/secret_env_2/` — elles peuvent être rejouées via `env.play_policy_step_by_step(pi)` sans réentraîner.